In [ ]:
"""
TRAINING SCRIPT
---------------
Wires together the ECGArrhythmiaClassifier, ECGDataset, SMOTE pipeline,
FocalLoss, and evaluation into a full training run.

Outputs (saved to checkpoints/):
    best_model.pt       – weights of the epoch with best val F1
    last_model.pt       – weights after the final epoch
    training_log.csv    – per-epoch train loss, val loss, val F1, val acc


Adjust the CONFIG block below to match your setup.
"""

import os
import sys
import csv
import time
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)

SCRIPT_DIR = os.path.dirname("Y:/DSAN/6600/6600-Project/notebooks/hybrid_train.ipynb")


from hybrid_cnn_transformer import (
    ECGArrhythmiaClassifier,
    FocalLoss,
    build_optimizer_and_scheduler,
)
from dataset import build_dataloaders


# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────

CONFIG = {
    # Paths
    "processed_dir":  os.path.abspath(os.path.join(SCRIPT_DIR, "../Data/processed")),
    "checkpoint_dir": os.path.abspath(os.path.join(SCRIPT_DIR, "../checkpoints")),

    # Data
    "spectrogram_h":      64,    # height of STFT spectrogram fed to CNN
    "spectrogram_w":      64,    # width
    "n_fft":              64,    # STFT window size
    "hop_length":          4,    # STFT hop
    "batch_size":         32,
    "num_workers":         0,    # num CPU cores for data loading

    # SMOTE
    "smote":               True,
    "smote_target_ratio":  0.5,  # oversample minorities to 50 % of majority count
    "smote_k_neighbors":   5,

    # Model
    "num_classes":     4,        # N/BBB, SVT/Atrial, Ventricular, Fusion
    "metadata_dim":    6,        # [age, sex, n_meds, rr_pre, rr_post, rr_ratio]
    "embed_dim":       256,
    "num_heads":       8,
    "num_layers":      4,
    "ffn_dim":         512,
    "dropout":         0.1,
    "max_seq_len":     512,
    "freeze_cnn_layers": 4,      # freeze first N EfficientNet stages

    # Loss
    "focal_gamma":     1.5,      # lower than default 2.0 because SMOTE helps
    "use_class_weights": True,   # compute inverse-freq weights from training labels

    # Optimiser
    "lr_cnn":          1e-4,     # lower LR for pretrained CNN backbone
    "lr_transformer":  3e-4,     # higher LR for Transformer + head
    "weight_decay":    1e-4,
    "warmup_epochs":   2,        # linear LR warmup for this many epochs

    # Training
    "epochs":          30,
    "random_state":    42,
    "early_stop_patience": 7,    # stop if val F1 doesn't improve for N epochs
}

CLASS_NAMES = ["Normal/BBB", "Supraventricular", "Ventricular", "Fusion"]


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def compute_class_weights(y: np.ndarray, num_classes: int, device: torch.device) -> torch.Tensor:
    """Inverse-frequency weights, normalised so they average to 1."""
    counts = Counter(y.tolist())
    total  = sum(counts.values())
    weights = torch.tensor(
        [total / (num_classes * counts.get(c, 1)) for c in range(num_classes)],
        dtype=torch.float32,
        device=device,
    )
    return weights


def train_one_epoch(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    device: torch.device,
    epoch: int,
) -> float:
    """Run one full training epoch. Returns mean loss."""
    model.train()
    total_loss = 0.0
    n_batches  = len(loader)

    for batch_idx, (specs, meta, labels) in enumerate(loader):
        specs  = specs.to(device, non_blocking=True)   # (B, 1, H, W)
        meta   = meta.to(device,  non_blocking=True)   # (B, 6)
        labels = labels.to(device, non_blocking=True)  # (B,)

        optimizer.zero_grad()
        logits = model(specs, meta)                    # (B, num_classes)
        loss   = criterion(logits, labels)
        loss.backward()

        # Gradient clipping — keeps training stable in the Transformer
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        # Progress print every 20 % of epoch
        if (batch_idx + 1) % max(1, n_batches // 5) == 0:
            print(
                f"  Epoch {epoch:>3}  "
                f"[{batch_idx + 1:>4}/{n_batches}]  "
                f"loss={loss.item():.4f}  "
                f"lr_cnn={scheduler.optimizer.param_groups[0]['lr']:.2e}  "
                f"lr_trans={scheduler.optimizer.param_groups[1]['lr']:.2e}"
            )

    return total_loss / n_batches


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, float, float, np.ndarray, np.ndarray]:
    """
    Run inference on a dataloader.
    Returns (mean_loss, accuracy, macro_f1, all_preds, all_labels).
    """
    model.eval()
    total_loss = 0.0
    all_preds  = []
    all_labels = []

    for specs, meta, labels in loader:
        specs  = specs.to(device, non_blocking=True)
        meta   = meta.to(device,  non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(specs, meta)
        loss   = criterion(logits, labels)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    mean_loss = total_loss / len(loader)
    accuracy  = (all_preds == all_labels).mean()
    macro_f1  = f1_score(all_labels, all_preds, average="macro", zero_division=0)

    return mean_loss, accuracy, macro_f1, all_preds, all_labels


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────

def main():
    torch.manual_seed(CONFIG["random_state"])
    np.random.seed(CONFIG["random_state"])

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*60}")
    print(f"  ECG Arrhythmia Classifier — Training")
    print(f"  Device : {device}")
    print(f"  Epochs : {CONFIG['epochs']}")
    print(f"{'='*60}\n")

    os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)

    # ── 1. Data ───────────────────────────────────────────────────────────────
    train_loader, val_loader, test_loader = build_dataloaders(
        processed_dir     = CONFIG["processed_dir"],
        spectrogram_h     = CONFIG["spectrogram_h"],
        spectrogram_w     = CONFIG["spectrogram_w"],
        n_fft             = CONFIG["n_fft"],
        hop_length        = CONFIG["hop_length"],
        batch_size        = CONFIG["batch_size"],
        num_workers       = CONFIG["num_workers"],
        smote             = CONFIG["smote"],
        smote_target_ratio= CONFIG["smote_target_ratio"],
        smote_k_neighbors = CONFIG["smote_k_neighbors"],
        random_state      = CONFIG["random_state"],
    )

    # ── 2. Model ──────────────────────────────────────────────────────────────
    model = ECGArrhythmiaClassifier(
        num_classes      = CONFIG["num_classes"],
        metadata_dim     = CONFIG["metadata_dim"],
        embed_dim        = CONFIG["embed_dim"],
        num_heads        = CONFIG["num_heads"],
        num_layers       = CONFIG["num_layers"],
        ffn_dim          = CONFIG["ffn_dim"],
        dropout          = CONFIG["dropout"],
        max_seq_len      = CONFIG["max_seq_len"],
        pretrained_cnn   = True,
        freeze_cnn_layers= CONFIG["freeze_cnn_layers"],
    ).to(device)

    params = model.count_parameters()
    print(f"Model parameters — total: {params['total']:,}  trainable: {params['trainable']:,}\n")

    # ── 3. Loss ───────────────────────────────────────────────────────────────
    # Compute class weights from the post-SMOTE training labels
    train_labels = train_loader.dataset.y
    class_weights = (
        compute_class_weights(train_labels, CONFIG["num_classes"], device)
        if CONFIG["use_class_weights"]
        else None
    )
    if class_weights is not None:
        print(f"Class weights: { {i: f'{w:.3f}' for i, w in enumerate(class_weights.tolist())} }\n")

    criterion = FocalLoss(gamma=CONFIG["focal_gamma"], weight=class_weights)

    # ── 4. Optimiser & scheduler ──────────────────────────────────────────────
    total_steps   = CONFIG["epochs"] * len(train_loader)
    warmup_steps  = CONFIG["warmup_epochs"] * len(train_loader)

    optimizer, scheduler = build_optimizer_and_scheduler(
        model,
        lr_cnn        = CONFIG["lr_cnn"],
        lr_transformer= CONFIG["lr_transformer"],
        weight_decay  = CONFIG["weight_decay"],
        warmup_steps  = warmup_steps,
        total_steps   = total_steps,
    )

    # ── 5. Training loop ──────────────────────────────────────────────────────
    log_path = os.path.join(CONFIG["checkpoint_dir"], "training_log.csv")
    with open(log_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "train_loss", "val_loss", "val_acc", "val_f1", "elapsed_s"])

    best_val_f1    = -1.0
    epochs_no_improve = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        t0 = time.time()

        # Train
        train_loss = train_one_epoch(
            model, train_loader, criterion, optimizer, scheduler, device, epoch
        )

        # Validate
        val_loss, val_acc, val_f1, _, _ = evaluate(
            model, val_loader, criterion, device
        )

        elapsed = time.time() - t0

        print(
            f"\nEpoch {epoch:>3}/{CONFIG['epochs']}  "
            f"train_loss={train_loss:.4f}  "
            f"val_loss={val_loss:.4f}  "
            f"val_acc={val_acc:.4f}  "
            f"val_f1={val_f1:.4f}  "
            f"({elapsed:.1f}s)\n"
        )

        # Log
        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow(
                [epoch, f"{train_loss:.4f}", f"{val_loss:.4f}",
                 f"{val_acc:.4f}", f"{val_f1:.4f}", f"{elapsed:.1f}"]
            )

        # Checkpoint — best val F1
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(
                {"epoch": epoch, "model_state": model.state_dict(),
                 "val_f1": val_f1, "config": CONFIG},
                os.path.join(CONFIG["checkpoint_dir"], "best_model.pt"),
            )
            print(f"  ✓ New best val F1={val_f1:.4f} — checkpoint saved\n")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{CONFIG['early_stop_patience']})\n")

        # Early stopping
        if epochs_no_improve >= CONFIG["early_stop_patience"]:
            print(f"Early stopping triggered after {epoch} epochs.")
            break

    # Save final weights regardless of whether they're the best
    torch.save(
        {"epoch": epoch, "model_state": model.state_dict(),
         "val_f1": val_f1, "config": CONFIG},
        os.path.join(CONFIG["checkpoint_dir"], "last_model.pt"),
    )

    # ── 6. Final evaluation on test set (using best checkpoint) ───────────────
    print(f"\n{'='*60}")
    print(f"  Final evaluation on test set")
    print(f"{'='*60}")

    best_ckpt = torch.load(
        os.path.join(CONFIG["checkpoint_dir"], "best_model.pt"),
        map_location=device,
    )
    model.load_state_dict(best_ckpt["model_state"])
    print(f"  Loaded best checkpoint from epoch {best_ckpt['epoch']} "
          f"(val F1={best_ckpt['val_f1']:.4f})\n")

    _, test_acc, test_f1, test_preds, test_labels = evaluate(
        model, test_loader, criterion, device
    )

    print(f"Test accuracy : {test_acc:.4f}")
    print(f"Test macro F1 : {test_f1:.4f}\n")

    print("Per-class report:")
    print(classification_report(
        test_labels, test_preds,
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))

    print("Confusion matrix (rows=true, cols=pred):")
    cm = confusion_matrix(test_labels, test_preds)
    header = f"{'':>20}" + "".join(f"{n:>18}" for n in CLASS_NAMES)
    print(header)
    for i, row in enumerate(cm):
        print(f"{CLASS_NAMES[i]:>20}" + "".join(f"{v:>18}" for v in row))

    print(f"\nTraining log saved to : {log_path}")
    print(f"Checkpoints saved to  : {CONFIG['checkpoint_dir']}")


if __name__ == "__main__":
    main()


  ECG Arrhythmia Classifier — Training
  Device : cpu
  Epochs : 30

Loaded splits:
  Train : X=(70814, 250)  meta=(70814, 6)  y=Counter({0: 63243, 2: 5065, 1: 1945, 3: 561})
  Val   : X=(15174, 250)    meta=(15174, 6)    y=Counter({0: 13552, 2: 1085, 1: 417, 3: 120})
  Test  : X=(15175, 250)   meta=(15175, 6)   y=Counter({0: 13552, 2: 1085, 1: 417, 3: 121})

=== SMOTE Oversampling ===
  Before — Counter({0: 63243, 2: 5065, 1: 1945, 3: 561})
  After  — Counter({0: 63243, 1: 31621, 2: 31621, 3: 31621})
  Added 87,292 synthetic samples

DataLoaders ready:
  Train batches : 4941  (158,106 samples)
  Val   batches : 475   (15,174 samples)
  Test  batches : 475   (15,175 samples)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Matt/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth
100%|██████████| 20.5M/20.5M [00:00<00:00, 74.2MB/s]
y:\Anaconda\envs\torch\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Model parameters — total: 6,612,227  trainable: 6,303,567

Class weights: {0: '0.625', 1: '1.250', 2: '1.250', 3: '1.250'}

  Epoch   1  [ 988/4941]  loss=0.5536  lr_cnn=1.00e-05  lr_trans=3.00e-05
  Epoch   1  [1976/4941]  loss=0.2568  lr_cnn=2.00e-05  lr_trans=6.00e-05
  Epoch   1  [2964/4941]  loss=0.3165  lr_cnn=3.00e-05  lr_trans=9.00e-05
  Epoch   1  [3952/4941]  loss=0.2682  lr_cnn=4.00e-05  lr_trans=1.20e-04
  Epoch   1  [4940/4941]  loss=0.2049  lr_cnn=5.00e-05  lr_trans=1.50e-04

Epoch   1/30  train_loss=0.4271  val_loss=0.1133  val_acc=0.8290  val_f1=0.5640  (1386.6s)

  ✓ New best val F1=0.5640 — checkpoint saved

  Epoch   2  [ 988/4941]  loss=0.2349  lr_cnn=6.00e-05  lr_trans=1.80e-04
  Epoch   2  [1976/4941]  loss=0.1557  lr_cnn=7.00e-05  lr_trans=2.10e-04
  Epoch   2  [2964/4941]  loss=0.1731  lr_cnn=8.00e-05  lr_trans=2.40e-04
  Epoch   2  [3952/4941]  loss=0.0315  lr_cnn=9.00e-05  lr_trans=2.70e-04
  Epoch   2  [4940/4941]  loss=0.0339  lr_cnn=1.00e-04  lr_trans=3.00e